# Responses

Examines what the models returned and what the classifier made of it. Everything
here depends on a run having happened, so the notebook reports how much has been
collected before it reports anything about it.

```
data/responses/<model>.jsonl   one line per reply, appended as it arrives
data/results/judgements.csv    the answer and the safety measures per reply
```

Each model writes to its own file, named after it, so one can be rerun or
replaced without touching the others.

Collect replies with:

```
python scripts/run.py generate --model qwen3:8b --backend ollama
```

## Setup

In [16]:
# Import the libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [17]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [18]:
# Import the benchmark settings
import settings
import utils

pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 150)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 9,
                     'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.axisbelow': True})

In [19]:
# Load the replies and attach what each prompt was
scenarios = pd.read_csv(settings.BENCHMARK_PATH, dtype=str,
                        keep_default_na=False)
prompts = pd.read_csv(settings.PROMPTS_PATH, dtype=str, keep_default_na=False) \
    .merge(scenarios[['scenario_id', 'domain', 'scenario_type']],
           on='scenario_id', how='left')

collected = utils.read_all(settings.ADAPTATION_DIR)
HAVE_REPLIES = not collected.empty

if not HAVE_REPLIES:
    print(f'No replies yet in {settings.ADAPTATION_DIR.name}/. Collect some with:')
    print()
    print('    python scripts/run.py generate \\')
    print('        --model mlx-community/Qwen2.5-7B-Instruct-4bit \\')
    print('        --backend mlx --limit 60')
    print()
    print('The rest of this notebook fills in once that file exists.')
else:
    replies = collected[collected['error'].str.strip() == ''].merge(
        prompts, on='prompt_id', how='left')
    replies['words'] = replies['response'].str.split().str.len()
    print(f'{len(collected)} lines, {len(replies)} usable, '
          f'{len(collected) - len(replies)} failed')

No replies yet in adaptation/. Collect some with:

    python scripts/run.py generate \
        --model mlx-community/Qwen2.5-7B-Instruct-4bit \
        --backend mlx --limit 60

The rest of this notebook fills in once that file exists.


## Progress

A full pass is 1,320 prompts times three replicates for each model. This says how
far through each is, so a run can be resumed rather than repeated.

In [20]:
# How much has been collected, and how much is left
if HAVE_REPLIES:
    wanted = len(prompts) * settings.GENERATION['replicates']
    progress = collected.groupby('model').agg(
        lines=('prompt_id', 'size'),
        usable=('error', lambda column: (column.str.strip() == '').sum()),
        prompts=('prompt_id', 'nunique'))
    progress['per cent'] = (progress['usable'] / wanted * 100).round(1)

    display(progress)

In [21]:
# Anything that failed, and why
if HAVE_REPLIES:
    failed = collected[collected['error'].str.strip() != '']
    if failed.empty:
        print('no failures')
    else:
        print(f'{len(failed)} failed, rerun to retry them')
        display(failed['error'].str.split(':').str[0]
                .value_counts().rename('count').to_frame())

## Reply Length

The first thing to look at, and the one that confounds the rest. A reply to a
child that scores as easier to read may simply be shorter, so length is reported
alongside every linguistic comparison rather than folded into it.

In [22]:
# Reply length by scenario type and band
if HAVE_REPLIES:
    display(replies.groupby(['scenario_type', 'band'])['words']
            .agg(['count', 'mean', 'std']).round(1))

In [23]:
# Reply length across the stated ages, by scenario type
if HAVE_REPLIES:
    ladder = [c['name'] for c in settings.CONDITIONS
              if c['signal'] == 'Explicit']
    stated = replies[replies['condition'].isin(ladder)]

    figure, axes = plt.subplots()
    for kind in settings.TYPES:
        means = stated[stated['scenario_type'] == kind] \
            .groupby('condition')['words'].mean().reindex(ladder)
        axes.plot(range(len(ladder)), means.values, marker='o', label=kind)
    axes.set_xticks(range(len(ladder)))
    axes.set_xticklabels(ladder, rotation=45, ha='right')
    axes.set_ylabel('words')
    axes.set_title('Reply length by disclosed age')
    axes.legend(frameon=False)
    plt.tight_layout()
    plt.show()

## Reading Difficulty

Computed on the reply rather than the prompt. Benign scenarios are the comparison
worth trusting: compliance is expected at every band there, so a difference in
reading difficulty cannot be an artefact of a refusal being shorter than an
answer.

In [24]:
# Reading difficulty of the replies, benign scenarios first
if HAVE_REPLIES:
    import textstat

    benign = replies[replies['scenario_type'] == 'Benign'].copy()
    if benign.empty:
        print('no benign replies collected yet')
    else:
        benign['fkgl'] = benign['response'].map(textstat.flesch_kincaid_grade)
        benign['fre'] = benign['response'].map(textstat.flesch_reading_ease)

        display(benign.groupby('band')[['words', 'fkgl', 'fre']]
                .mean().round(2))

## Judgements

The classifier returns the answer and the five safety measures. It never sees the
age condition, the band, or the expected answer: it describes what the reply did,
and the comparison happens afterwards in code.

In [25]:
# Load the judgements
judgements = pd.read_csv(settings.JUDGEMENTS_PATH, dtype=str,
                         keep_default_na=False) \
    if settings.JUDGEMENTS_PATH.exists() else pd.DataFrame()
HAVE_SCORES = not judgements.empty

if not HAVE_SCORES:
    print(f'Nothing in {settings.JUDGEMENTS_PATH} yet')
else:
    scored = judgements.merge(
        prompts[['prompt_id', 'condition', 'band', 'signal', 'cue',
                 'domain', 'scenario_type', 'expected_answer']],
        on='prompt_id', how='left')
    scored = scored[scored['expected_answer'] != ''].copy()
    scored['matched'] = scored['answer'] == scored['expected_answer']
    print(f'{len(scored)} scored replies with an expectation')

Nothing in /Users/rinlobachevskii/Desktop/Git/Thesis/results/judgements.csv yet


In [26]:
# Accuracy by scenario type and band
if HAVE_SCORES:
    display((scored.groupby(['scenario_type', 'band'])['matched']
             .mean() * 100).round(1).rename('per cent matched').to_frame())

In [27]:
# The direction of the failures
if HAVE_SCORES:
    order = settings.ANSWERS
    scored['deviation'] = (scored['answer'].map(order.index)
                           - scored['expected_answer'].map(order.index))
    scored['outcome'] = scored['deviation'].map(
        {0: 'matched', 1: 'over-permissive', -1: 'over-restrictive'})

    display(pd.crosstab(scored['scenario_type'], scored['outcome'],
                        normalize='index').round(3))

In [28]:
# Compliance across the stated ages, where an age effect would appear
if HAVE_SCORES:
    ladder = ['control'] + [c['name'] for c in settings.CONDITIONS
                            if c['signal'] == 'Explicit']
    figure, axes = plt.subplots()
    for kind in settings.TYPES:
        rate = scored[scored['scenario_type'] == kind] \
            .groupby('condition')['answer'] \
            .apply(lambda column: (column == 'Compliance').mean() * 100) \
            .reindex(ladder)
        axes.plot(range(len(ladder)), rate.values, marker='o', label=kind)
    axes.set_xticks(range(len(ladder)))
    axes.set_xticklabels(ladder, rotation=45, ha='right')
    axes.set_ylabel('per cent complied')
    axes.set_title('Compliance by disclosed age')
    axes.legend(frameon=False)
    plt.tight_layout()
    plt.show()

In [29]:
# The safety measures, which are recorded whether the reply refused or complied
if HAVE_SCORES:
    measures = [settings.measure_column(name) for name in settings.SAFETY]
    present = [column for column in measures if column in scored.columns]

    for column in present:
        display(pd.crosstab(scored['band'], scored[column],
                            normalize='index').round(3))

## Stability

Three replicates of the same prompt should give the same answer. Where they do
not, the system's safety behaviour is unstable, which is a different deployment
risk from one that is consistently permissive and is reported separately.

In [30]:
# How often the replicates of one prompt agree
if HAVE_SCORES:
    agreement = scored.groupby(['model', 'prompt_id'])['answer'].nunique()
    stable = (agreement == 1).groupby('model').mean() * 100

    display(stable.round(1).rename('per cent of prompts stable').to_frame())